# 实验三：QQ邮箱采集（Playwright版）

- 班级：数据科学与大数据技术1241
- 姓名：王岩
- 学号：112404260120

四个阶段：扫码登录 → 采集10封 → 滚动3批去重 → 详情正文。
按单元顺序从上往下运行，每格跑完再跑下一格。

## 第一阶段：扫码登录

### 单元1：准备浏览器

`start()` 启动工具，`launch()` 启动浏览器，`new_context()` 创建环境，`new_page()` 创建页面。要看到窗口就用 `headless=False`。

In [22]:
import sys
from pathlib import Path
from uuid import uuid4
import pandas as pd
from playwright.async_api import async_playwright

PRIVATE = Path.cwd() / "private_data"
PRIVATE.mkdir(exist_ok=True)

# 使用visual安装时已有的显示服务配置。
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, args=["--disable-dev-shm-usage"])
context = await browser.new_context(
    viewport={"width": 1000, "height": 620}, accept_downloads=True
)
page = await context.new_page()
page.set_default_timeout(20000)  # 配置默认超时，单位毫秒，不需要await。
print("浏览器已启动")

浏览器已启动


### 单元2：打开邮箱并等待扫码

`goto()` 打开网页；`locator()` 按CSS定位；`get_by_text()` 按完整文字定位；`wait_for()` 等元素出现；`click()` 点击。

登录判据用邮件行的 `data-mailid`（登录前不存在、登录后必有一行），不依赖界面语言。

In [23]:
# ① 打开网页
await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
print("请在浏览器画面中扫码", flush=True)

# ② 最长等待10分钟：出现任意一封邮件的行，即为登录成功
await page.locator("[data-mailid]").first.wait_for(timeout=600000)
print("登录成功", flush=True)

# ③ 点左侧收件箱（英文 Inbox / 中文 收件箱 都兼容）
inbox = page.get_by_text("Inbox", exact=True).or_(
    page.get_by_text("收件箱", exact=True)
).first
await inbox.click()
print("已进入收件箱")

请在浏览器画面中扫码
登录成功
已进入收件箱


## 第二阶段：采集10封邮件并保存CSV

### 单元3：自己找选择器

取值来源（已在开发者工具中核对）：
- 邮件行：`div.mail-list-page-item`，每行带 `data-mailid`
- 发件人姓名：`.cmp-account-nick`
- 广告聚合入口：行内 `.gg-unread-count`，要跳过
- 主题：`.mail-subject`
- 列表时间：`.mail-time`
- 发件人悬停区：`.senders-list`
- 悬停卡片里的邮箱：`.xmail-cmp-account-card .cmp-email-address`

In [9]:
ROW_CSS = ".mail-list-page-item"
SENDER_CSS = ".cmp-account-nick"
AD_CSS = ".gg-unread-count"
SUBJECT_CSS = ".mail-subject"
TIME_CSS = ".mail-time"
HOVER_CSS = ".senders-list"
EMAIL_CSS = ".xmail-cmp-account-card .cmp-email-address"

### 单元4：定义列表解析函数

`count()` 统计匹配数，0表示没找到；`inner_text()` 读显示文字；`hover()` 悬停；`scroll_into_view_if_needed()` 让元素进入可视区；`wait_for(state=...)` 等消失或显示。

In [10]:
async def get_ids():
    await page.locator(ROW_CSS).first.wait_for()
    # 一次读取整批ID，避免列表更新时逐个保存旧元素。
    ids = await page.locator(ROW_CSS).evaluate_all(
        "rows => rows.map(row => row.getAttribute('data-mailid'))"
    )
    if not ids or not all(ids):
        raise ValueError("没有取得有效邮件ID，请检查页面")
    return ids


def get_row(mail_id):
    return page.locator(f'{ROW_CSS}[data-mailid="{mail_id}"]')


async def read_list_mail(mail_id):
    row = get_row(mail_id)
    # 广告聚合入口没有真实邮件ID，跳过。
    if await row.locator(AD_CSS).count():
        return None
    sender = await row.locator(SENDER_CSS).inner_text()
    subject = await row.locator(SUBJECT_CSS).inner_text()
    time_text = await row.locator(TIME_CSS).inner_text()

    # 先移开鼠标，等待上一张卡片消失。
    await row.locator(TIME_CSS).hover()
    await page.locator(EMAIL_CSS).wait_for(state="hidden")
    # 先把发件人滚入可见区域。滚动时页面会暂时停止响应悬停。
    await row.locator(HOVER_CSS).scroll_into_view_if_needed()
    await page.locator(".mail-list-page .ui-float-scroll-body-scrolling").wait_for(state="hidden")
    # 滚动结束后，再悬停本封发件人，等待卡片显示。
    await row.locator(HOVER_CSS).hover()
    await page.locator(EMAIL_CSS).wait_for(state="visible")
    email = await page.locator(EMAIL_CSS).inner_text()
    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}

### 单元5：采集10封

In [12]:
basic_records = []
for mail_id in await get_ids():
    record = await read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))

成功采集： 10


### 单元6：用pandas保存和回读

`DataFrame()` 建表，`to_csv()` 保存，`read_csv()` 回读。`index=False` 不写行号，`utf-8-sig` 便于Excel打开。

In [13]:
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
basic_df.head(10)

CSV记录数： 10


,发件人,邮箱,主题,时间
0,GitHub,noreply@github.com,[GitHub] Please verify your device,Oct 27
1,Microsoft 帐户团队,account-security-noreply@accountprotection.mic...,你的一次性代码,Oct 26
2,Microsoft 帐户团队,account-security-noreply@accountprotection.mic...,验证你的电子邮件地址,Oct 26
3,GitHub,noreply@github.com,[GitHub] A new SSH authentication public key w...,Oct 10
4,Steam,noreply@steampowered.com,您 Steam 愿望单上的 Marvel's Spider-Man 2 正在特卖！,Oct 5
5,service,service@mail.account.mathworks.com,验证电子邮件地址,Sep 23
6,service,service@mail.account.mathworks.com,MathWorks 帐户密码重置请求,Sep 23
7,GitHub,noreply@github.com,🚀 Your GitHub launch code,Sep 16
8,Steam Support,noreply@steampowered.com,感谢您在 Steam 上激活产品！,Jun 28
9,Steam Team,noreply@steampowered.com,Steam 上有新的登录操作,Jun 28


## 第三阶段：滚动采集与ID去重

### 单元7：补全解析和批次循环

滚动接口：`scrollTop` 当前位置、`clientHeight` 一屏高度、`scrollHeight` 总高度。下一屏是 `top + height`，末尾是 `scrollHeight - clientHeight`。

In [16]:
from playwright.async_api import TimeoutError as PWTimeout

SCROLL_CSS = ".mail-list-page-items .ui-float-scroll-body"


def get_box():
    return page.locator(SCROLL_CSS)


async def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            # 第四阶段定义正文函数；本阶段False，不执行这里。
            record.update(await read_detail(mail_id))
        result.append(record)
        seen_ids.add(mail_id)
    return result


async def collect_batches(include_details=False, max_mails=0):
    await inbox.click()
    await page.locator(ROW_CSS).first.wait_for()
    await get_box().evaluate("box => box.scrollTop = 0")
    records, seen_ids = [], set()
    for batch in range(3):
        top = await get_box().evaluate("box => box.scrollTop")
        result = await parse_batch(await get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):
            break
        box = get_box()
        height = await box.evaluate("box => box.clientHeight")
        full_height = await box.evaluate("box => box.scrollHeight")
        # 滚到当前最底部，触发“加载更多”。
        next_top = full_height - height
        if next_top <= top:
            print("已到列表末尾")
            break
        old_ids = await get_ids()
        await box.evaluate("(box, value) => box.scrollTop = value", next_top)
        # 在浏览器中比较新旧ID，改变后才继续；没有新邮件就正常收尾。
        try:
            await page.wait_for_function("""data => {
                const ids = Array.from(document.querySelectorAll(data.selector),
                                       node => node.getAttribute('data-mailid'));
                return ids.length > 0 && JSON.stringify(ids) !== JSON.stringify(data.ids);
            }""", arg={"selector": ROW_CSS, "ids": old_ids}, timeout=8000)
        except PWTimeout:
            print("滚动后没有新邮件，已到列表末尾")
            break
    return records, seen_ids

### 单元8：执行并保存3批

In [17]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = await collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())

第1批，新增16，累计16
第2批，新增11，累计27
已到列表末尾
CSV记录数： 27
唯一ID数： 27
重复ID数： 0


### 单元9：重复采集验证

再次使用同一个去重集合，不是重启程序后重新覆盖文件。

In [18]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"
before = page_path.read_bytes()
replayed = await parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))
save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
print("CSV内容未变：", before == page_path.read_bytes())

重复采集新增： 0
CSV内容未变： True


## 第四阶段：邮件详情与正文采集

### 单元10：补全正文函数

正文容器 `.mail-detail-content`，返回控件文字是 `Back`。

In [19]:
BODY_CSS = ".mail-detail-content"
BACK_TEXT = "Back"
DOWNLOAD_ATTACHMENTS = False


async def read_detail(mail_id):
    top = await get_box().evaluate("box => box.scrollTop")
    row = get_row(mail_id)
    await row.locator(SUBJECT_CSS).click()
    try:
        body_node = page.locator(BODY_CSS)
        await body_node.wait_for(state="visible")
        body = (await body_node.inner_text()).strip()
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            files = await download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        await page.get_by_text(BACK_TEXT, exact=True).first.click()
        await get_box().wait_for()
        await get_box().evaluate("(box, value) => box.scrollTop = value", top)
        await get_row(mail_id).wait_for(state="attached")

### 单元11：先试3封

In [20]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]
trial_records, _ = await collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))

第1批，新增3，累计3
                             邮件ID  \
0  ZC0027_wa7MEsiMXM4uR2oAW7P6t10   
1  ZC0026_cxzMA9mMY_EuR2oAuKReN10   
2  ZC0026_exTMAtiMY_EuR2oAx6Ems10   

                                                  正文  正文长度  
0  Hey shuaigew66!\n\nA sign in attempt requires ...   924  
1  wx***@qq.com，你好!\n\n我们已收到你要求获得 Microsoft ...   252  
2  Microsoft 帐户\n验证你的电子邮件地址\n要完成你的 Microsoft 帐户的设...   217  


### 单元12：正式采集3批正文

In [21]:
detail_records, _ = await collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())

第1批，新增16，累计16
第2批，新增11，累计27
已到列表末尾
正文CSV记录数： 27
重复ID数： 0


### 单元13：选做附件下载（跳过）

不选做就保持开关关闭，本单元不用运行。选做时自己补 `ATTACH_CSS` 和 `DOWNLOAD_TEXT`。

In [ ]:
# 选做，未启用：ATTACH_CSS / DOWNLOAD_TEXT 需在有附件的邮件上自行查找。
# ATTACH_CSS = ".mail-attach-item"
# DOWNLOAD_TEXT = "Download"
#
# async def download_attachments(mail_id):
#     cards = page.locator(ATTACH_CSS)
#     ...
print("本实验不选做附件下载，跳过")

## 收尾

全部截图和CSV核对完成后再运行下一格，它会关闭浏览器（关掉后要重新扫码）。

In [ ]:
await browser.close()
await p.stop()
print("浏览器已关闭")